# Manual Annotation of Detections

Listen to every exported detection clip and mark it **Call** (a genuine target
call), **False positive**, or **Unsure**. Labels are saved to a single CSV as
you go (`data/outputs/annotations/detection_labels.csv`), so you can stop and
resume any time — already-labeled clips are skipped.

When you finish (or at any point), the last section prints the numbers the
paper's field-deployment paragraph needs: per-station detection counts, Cernic
detections kept after review, Colobus false positives, and precision.

**Run Step 5 of `main_local.ipynb` first** so the clips exist under
`data/outputs/detected_clips/`. You also need `ipywidgets` (`pip install ipywidgets`).


## 1 — Set up and scan the clips


In [ ]:
import os, sys, importlib
from pathlib import Path

def find_repo_root(start=None):
    p = Path(start or os.getcwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / 'src' / 'config.py').is_file() and (cand / 'data').is_dir():
            return cand
    raise RuntimeError('Run this notebook from inside the cloned repo.')

REPO = find_repo_root()
os.environ['PRIMATE_DATA_ROOT'] = str(REPO / 'data')
SRC = str(REPO / 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import config, annotation
importlib.reload(config); importlib.reload(annotation)

# Where Step 5 wrote the review clips, and where labels are stored.
CLIPS_DIR = annotation.default_clips_dir()          # data/outputs/detected_clips
CSV_PATH  = annotation.default_annotation_csv()     # data/outputs/annotations/detection_labels.csv

clips = annotation.scan_clips(CLIPS_DIR)
ann   = annotation.load_annotations(CSV_PATH)
done, total = annotation.progress(clips, ann)
print(f'Clips found : {total}   (under {CLIPS_DIR})')
print(f'Already labeled: {done}   Remaining: {total - done}')
if total == 0:
    print('\nNo clips. Run Step 5 of main_local.ipynb first, or set CLIPS_DIR '
          'to your detected_clips folder.')
clips.head()


## 2 — Annotate

For each clip you'll see its **species / station / recording / start-time /
confidence**, a **spectrogram**, and an **audio player**. Press play, listen,
then click one of:

- **✓ Call** — a genuine call of the labeled species (true positive)
- **✗ False positive** — not the target (bird, insect, wind, other)
- **? Unsure** — revisit later

Use **◀ Back** to fix the previous clip. Every click saves immediately.


In [ ]:
import ipywidgets as W
from IPython.display import display, Audio, clear_output
import numpy as np, librosa, librosa.display
import matplotlib.pyplot as plt

class ClipAnnotator:
    def __init__(self, clips_df, csv_path):
        self.clips = clips_df.reset_index(drop=True)
        self.csv_path = csv_path
        self.ann = annotation.load_annotations(csv_path)
        labeled = set(self.ann['clip_id'])
        order = list(self.clips['clip_id'])
        self.i = next((k for k, c in enumerate(order) if c not in labeled), 0)
        self._build()

    def _build(self):
        self.status = W.HTML()
        self.info   = W.HTML()
        self.spec   = W.Output()
        self.audio  = W.Output()
        b_call = W.Button(description='\u2713 Call', button_style='success')
        b_fp   = W.Button(description='\u2717 False positive', button_style='danger')
        b_uns  = W.Button(description='? Unsure', button_style='warning')
        b_back = W.Button(description='\u25c0 Back')
        b_next = W.Button(description='Skip \u25b6')
        b_call.on_click(lambda _: self._label(annotation.TRUE_CALL))
        b_fp.on_click(lambda _: self._label(annotation.FALSE_POS))
        b_uns.on_click(lambda _: self._label(annotation.UNSURE))
        b_back.on_click(lambda _: self._move(-1))
        b_next.on_click(lambda _: self._move(1))
        controls = W.HBox([b_call, b_fp, b_uns, b_back, b_next])
        display(W.VBox([self.status, self.info, self.spec, self.audio, controls]))
        self._render()

    def _cur(self):
        return self.clips.iloc[self.i]

    def _label(self, label):
        self.ann = annotation.save_annotation(self._cur()['clip_id'], label,
                                              csv_path=self.csv_path)
        self._move(1)

    def _move(self, step):
        self.i = min(max(self.i + step, 0), len(self.clips) - 1)
        self._render()

    def _render(self):
        row = self._cur()
        prior = self.ann[self.ann['clip_id'] == row['clip_id']]
        prior_lbl = prior['label'].iloc[0] if len(prior) else '\u2014'
        done, total = annotation.progress(self.clips, self.ann)
        self.status.value = (f"<div style='font-size:14px'><b>{done}/{total} labeled</b>"
                             f" &nbsp;|&nbsp; viewing clip {self.i + 1} of {total}</div>")
        conf = row['confidence']
        conf_s = f"{conf:.3f}" if conf == conf else 'n/a'
        self.info.value = (
            "<div style='font-size:15px;line-height:1.5'>"
            f"<b>Species:</b> {row['species']} &nbsp;&nbsp; <b>Station:</b> {row['station']}<br>"
            f"<b>Recording:</b> {row['recording']} &nbsp;&nbsp; <b>Start:</b> {row['start_s']} s"
            f" &nbsp;&nbsp; <b>Model confidence:</b> {conf_s}<br>"
            f"<b>Current label:</b> <code>{prior_lbl}</code></div>")
        with self.spec:
            clear_output(wait=True)
            try:
                y, sr = librosa.load(row['path'], sr=None)
                S = librosa.amplitude_to_db(
                    np.abs(librosa.stft(y, n_fft=1024, hop_length=256)), ref=np.max)
                fig, ax = plt.subplots(figsize=(7, 2.4))
                librosa.display.specshow(S, sr=sr, hop_length=256, x_axis='time',
                                         y_axis='hz', ax=ax, cmap='magma')
                ax.set_ylim(0, 8000)
                ax.axhline(1500, color='cyan', lw=0.8, ls='--')  # 1.5 kHz Colobus gate line
                plt.tight_layout(); plt.show()
            except Exception as e:
                print('spectrogram error:', e)
        with self.audio:
            clear_output(wait=True)
            try:
                display(Audio(row['path']))
            except Exception as e:
                print('audio error:', e)

if len(clips):
    ClipAnnotator(clips, CSV_PATH)
else:
    print('No clips to annotate.')


## 3 — Summary for the paper

Re-run this cell any time to see the current tallies. It also writes
`per_species_summary.csv` and `per_station_summary.csv` next to the labels.


In [ ]:
importlib.reload(annotation)
clips = annotation.scan_clips(CLIPS_DIR)
ann   = annotation.load_annotations(CSV_PATH)

print(annotation.report_text(clips, ann))

summary = annotation.summarize(clips, ann)
out_dir = os.path.dirname(CSV_PATH)
summary['per_species'].to_csv(os.path.join(out_dir, 'per_species_summary.csv'))
summary['per_station'].to_csv(os.path.join(out_dir, 'per_station_summary.csv'))
summary['true_by_station'].to_csv(os.path.join(out_dir, 'kept_calls_by_station.csv'))
print('\nSaved per-species / per-station summaries to', out_dir)
print('\nDetections per station (all species):')
display(summary['per_station'])
print('Confirmed calls kept per station:')
display(summary['true_by_station'])
